### Project 1: An AI-powered marketing brochures generator
Used technology:
- OpenAI API call
- One-shot prompting
- Streaming results

Most parts in this file is adapted from course material from:
"AI Engineer Core Track: LLM Engineering, RAG, QLoRA, Agent" by "Ed Donner"
Orginal concepts and base implementation belongs to the course author 

Modifications made in this projects is for self-experimentation:
- Different LLMs model
- Modified system and user prompt 

This adaptation is for learning and project development purposes

In [2]:
# Import necessary libraries
import os
import json
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [12]:
# Initialize 
MODEL = "llama3.2"
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ignore")

### First, use model to search for relevant link

In [66]:
# Construct system prompt using single shot prompting
links_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages, Contact pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [77]:

# def get_links_user_prompt(url):
#     user_prompt = f"""
# Here is the list of links on the website {url}-
# Your task is to decide which of these are the relevant web links for a brochure about the company,
# Please response the full https URL in JSON format.
# Do not include Terms of Service, Privacy and email links.

# Links (might contain relative links):

# """
#     links = fetch_website_links(url)
#     user_prompt += "\n".join(links)
#     return user_prompt


def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

def get_links_messages(url):
    message = [
        {"role": "system", "content": links_system_prompt}, 
        {"role": "user", "content": get_links_user_prompt(url)}
    ]

    return message

In [79]:
def select_relevant_links(url):
    print(f"Using model {MODEL} to select relevant link")
    response = openai.chat.completions.create(
        model = "llama3.2",
        messages= get_links_messages(url),
        response_format={"type": "json_object"}
    )

    results = response.choices[0].message.content
    links = json.loads(results)
    print(f"Found {len(links["links"])} links:")
    return links

In [80]:
# Testing url "https://huggingface.co"
select_relevant_links("https://huggingface.co")

Using model llama3.2 to select relevant link
Found 7 links:


{'links': [{'type': 'home page', 'url': 'https://huggingface.co'},
  {'type': 'learn more about the company',
   'url': 'https://apply.workable.com/hugglingface/'},
  {'type': 'GitHub repository for Hugging Face',
   'url': 'https://github.com/huggingface'},
  {'type': 'Twitter account for Hugging Face',
   'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page for Hugging Face',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'official blog of Hugging Face',
   'url': 'https://blog.huggingface.co/'},
  {'type': 'Discord server for Hugging Face community',
   'url': 'https://join.huggingface.co/discord'}]}

#### Next, make the brochure


def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return resu